TESTING WITH SMALL BATCH OF CORPUS

In [ ]:

#Notebook setup
from pathlib import Path
from dotenv import load_dotenv
import os
import sys
import importlib

PROJECT_ROOT = Path(
    "/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()

load_dotenv(PROJECT_ROOT / ".env", override=True)

print("Project:", os.getcwd())
print("Nebius configured:", bool(os.getenv("NEBIUS_API_KEY")))
print("Neo4j configured:", bool(os.getenv("NEO4J_PASSWORD")))

Project: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag
Nebius configured: True
Neo4j configured: True


In [ ]:
#Install the additional LlamaIndex integrations
%pip install -U \
    llama-index-graph-stores-neo4j \
    llama-index-llms-openai-like \
    llama-index-embeddings-huggingface \
    sentence-transformers

In [ ]:

#Configure our Nebius LLM
from llama_index.llms.openai_like import OpenAILike
import os

llm = OpenAILike(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    api_base="https://api.tokenfactory.nebius.com/v1",
    api_key=os.getenv("NEBIUS_API_KEY"),

    is_chat_model=True,
    is_function_calling_model=False,

    # IMPORTANT:
    # Use response_format / JSON schema instead of tool calling
    should_use_structured_outputs=True,

    temperature=0.0,
    max_tokens=2000,
    context_window=16384,
)

print("Model:", llm.model)
print("Function calling:", llm.metadata.is_function_calling_model)
print("Structured outputs:", llm.should_use_structured_outputs)

Model: Qwen/Qwen3-30B-A3B-Instruct-2507
Function calling: False
Structured outputs: True


In [30]:
import importlib
import graph.schema

importlib.reload(graph.schema)

print(graph.schema.__file__)

/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/graph/schema.py


In [ ]:
#Define the graph schema
from typing import Literal
#Import constants
from graph.schema import (
    ENTITY_NAMES,
    RELATIONSHIP_NAMES,
    RELATIONSHIP_SCHEMAS,
)

# LlamaIndex SchemaLLMPathExtractor normalizes types to uppercase
LLAMAINDEX_ENTITY_NAMES = [
    entity.upper()
    for entity in ENTITY_NAMES
]

LLAMAINDEX_RELATIONSHIP_NAMES = [
    relationship.upper()
    for relationship in RELATIONSHIP_NAMES
]

LLAMAINDEX_VALIDATION_SCHEMA = [
    (
        source.upper(),
        relationship.upper(),
        target.upper(),
    )
    for source, relationship, target in RELATIONSHIP_SCHEMAS
]

#convert to Literal only for LlamaIndex
EntityTypes = Literal[tuple(LLAMAINDEX_ENTITY_NAMES)]
RelationTypes = Literal[tuple(LLAMAINDEX_RELATIONSHIP_NAMES)]

In [ ]:
from typing import get_args

print("Entities:")
print(get_args(EntityTypes))

print("\nRelationships:")
print(get_args(RelationTypes))

print("\nValidation schema:")
for x in LLAMAINDEX_VALIDATION_SCHEMA:
    print(x)

In [ ]:
#Verify that the LlamaIndex schema matches the original schema
from typing import get_args

print(get_args(EntityTypes))
print(get_args(RelationTypes))

('PERSON', 'TEAM', 'PROJECT', 'SKILL', 'TECHNOLOGY', 'DECISION', 'MEETING', 'DOCUMENT')
('AFFECTS', 'APPROVED', 'ATTENDED', 'AUTHORED', 'DESCRIBES', 'DISCUSSED', 'DOCUMENTS', 'HAS_SKILL', 'INVOLVES', 'LEADS', 'MEMBER_OF', 'MENTIONS', 'OWNED_BY', 'PROPOSED', 'USES', 'WORKED_ON')


In [50]:
print(type(ENTITY_NAMES))
print(EntityTypes)

<class 'list'>
typing.Literal['PERSON', 'TEAM', 'PROJECT', 'SKILL', 'TECHNOLOGY', 'DECISION', 'MEETING', 'DOCUMENT']


In [ ]:
#Prompt for extracting knowledge graph relationships from organizational documents
from llama_index.core import PromptTemplate

ORG_KG_EXTRACTION_PROMPT = PromptTemplate(
    """
Extract ALL explicitly stated knowledge graph relationships from the text
according to the provided schema.

GENERAL RULES

1. Extract every explicitly supported relationship.
   Do not summarize, rank, or omit relationships just because another
   relationship seems more important.

2. When a field contains a list, create one relationship for EVERY item.

3. Do not infer facts that are not explicitly stated.

4. Use only entity and relationship types allowed by the schema.

5. Use human-readable names from the document text.
   Never use metadata IDs such as entity_id, doc_id, project_atlas,
   decision_d001, or meeting_m001 as entity names.

   Example:
       entity_id: project_atlas
       heading: Project Atlas

   Extract:
       PROJECT name = "Project Atlas"

   not:
       PROJECT name = "project_atlas"


PROJECT DOCUMENT RULES

- "Project lead: <person>"
    -> PERSON --LEADS--> PROJECT

- Every person listed under "Contributors"
    -> PERSON --WORKED_ON--> PROJECT

- "Owning team: <team>"
    -> PROJECT --OWNED_BY--> TEAM

- Every item under "Core technologies"
    -> PROJECT --USES--> TECHNOLOGY


PERSON PROFILE RULES

- "Team: <team>"
    -> PERSON --MEMBER_OF--> TEAM

- Every item listed under "Skills"
    -> PERSON --HAS_SKILL--> SKILL

- Every item listed under "Projects"
    -> PERSON --WORKED_ON--> PROJECT

IMPORTANT:
A skill and a technology are different entity types.

Example:
If a person's Skills field contains Kafka:
    PERSON --HAS_SKILL--> SKILL("Kafka")

If a project's Core technologies field contains Kafka:
    PROJECT --USES--> TECHNOLOGY("Kafka")

Do not convert a person's skill into a TECHNOLOGY entity.


DECISION DOCUMENT RULES

- "Proposed by: <person>"
    -> PERSON --PROPOSED--> DECISION

- "Approved by: <person>"
    -> PERSON --APPROVED--> DECISION

- "Project: <project>"
    -> DECISION --AFFECTS--> PROJECT

- "Technology: <technology>"
    -> DECISION --INVOLVES--> TECHNOLOGY

Use the human-readable decision title as the DECISION entity name.


MEETING DOCUMENT RULES

- Use the meeting heading/title as a MEETING entity.

- Every person listed under "Participants"
    -> PERSON --ATTENDED--> MEETING

- Every decision listed under "Decisions discussed"
    -> MEETING --DISCUSSED--> DECISION


DOCUMENT RULES

Only create Document relationships when explicitly supported by the
document and schema. Do not invent authorship or documentation links.


Extract up to {max_triplets_per_chunk} relationships.

-------
{text}
-------
"""
)

print("✅ Organizational KG extraction prompt updated")

✅ Organizational KG extraction prompt updated


In [ ]:
#Create the schema-aware extractor
kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    extract_prompt=ORG_KG_EXTRACTION_PROMPT,

    possible_entities=EntityTypes,
    possible_relations=RelationTypes,
    kg_validation_schema=LLAMAINDEX_VALIDATION_SCHEMA,

    strict=True,
    allow_additional_properties=False,

    max_triplets_per_chunk=15,
    num_workers=1,
    raise_on_error=True,
)

print("✅ KG extractor recreated")

✅ KG extractor recreated


In [ ]:
#Test graph status
driver = get_driver()

with driver.session() as session:
    result = session.run("""
        MATCH (n)
        RETURN count(n) AS count
    """)

    print(
        "Nodes before clearing:",
        result.single()["count"]
    )

driver.close()

Nodes before clearing: 17


In [ ]:
#Clear graph for testing
driver = get_driver()

with driver.session() as session:
    session.run("""
        MATCH (n)
        DETACH DELETE n
    """)

driver.close()

print("✅ Test graph cleared")

✅ Test graph cleared


In [89]:
print("LLM class:", type(kg_extractor.llm))
print(
    "Function calling:",
    kg_extractor.llm.metadata.is_function_calling_model
)
print(
    "Structured outputs:",
    kg_extractor.llm.should_use_structured_outputs
)

LLM class: <class 'llama_index.llms.openai_like.base.OpenAILike'>
Function calling: False
Structured outputs: True


In [ ]:
#Connect LlamaIndex directly to Neo4j
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

graph_store = Neo4jPropertyGraphStore(
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    url=os.getenv("NEO4J_URI"),
    database=os.getenv("NEO4J_DATABASE", "neo4j"),
)

print("✅ Neo4j PropertyGraphStore created")

✅ Neo4j PropertyGraphStore created


In [ ]:
#Check whether Neo4j already contains anything
from graph.neo4j_client import get_driver

driver = get_driver()

with driver.session() as session:
    result = session.run(
        """
        MATCH (n)
        RETURN count(n) AS node_count
        """
    )
    print("Existing Neo4j nodes:", result.single()["node_count"])

driver.close()

Existing Neo4j nodes: 0


In [ ]:
#Load a test document for extraction
from llama_index.core import SimpleDirectoryReader

test_file = (
    PROJECT_ROOT
    / "data"
    / "projects"
    / "project_atlas.md"
)

test_documents = SimpleDirectoryReader(
    input_files=[str(test_file)]
).load_data()

print("Documents loaded:", len(test_documents))
print(test_documents[0].text)

Documents loaded: 1
---
doc_id: doc_project_atlas
doc_type: project_overview
entity_id: project_atlas
owner: Search
---
# Project Atlas

**Status:** Production  
**Owning team:** Search  
**Project lead:** Alice Chen  
**Core technologies:** Kafka, Elasticsearch, Python, Kubernetes

## Purpose
Modernize product search ranking with fresh behavioral signals and a unified retrieval stack.

## Contributors
Alice Chen, Bob Singh, George Liu, Hannah Brooks



In [ ]:
#Configure free local embeddings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print("✅ Local embedding model ready")

/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Langchain Basics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17348.41it/s]


✅ Local embedding model ready


In [19]:
%pip install -U nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#patch the running loop with nest_asyncio
import nest_asyncio

nest_asyncio.apply()

print("✅ asyncio patched for Jupyter")

✅ asyncio patched for Jupyter


In [ ]:
#first real extraction call
from llama_index.core import PropertyGraphIndex

index = PropertyGraphIndex.from_documents(
    test_documents,
    kg_extractors=[kg_extractor],
    embed_model=embed_model,
    property_graph_store=graph_store,
    show_progress=True,
    use_async=False,
)

print("✅ Atlas graph extraction completed")

Generating embeddings: 100%|██████████| 20/20 [00:00<00:00, 68.93it/s]


✅ Atlas graph extraction completed


In [ ]:
#Check the graph contents
driver = get_driver()

with driver.session() as session:

    result = session.run("""
        MATCH (n)
        RETURN
            labels(n) AS labels,
            count(*) AS count
        ORDER BY count DESC
    """)

    for record in result:
        print(
            record["labels"],
            record["count"]
        )

driver.close()

['__Node__', '__Entity__', 'PERSON'] 4
['__Node__', '__Entity__', 'TECHNOLOGY'] 4
['__Node__', 'Chunk'] 1
['__Node__', '__Entity__', 'PROJECT'] 1
['__Node__', '__Entity__', 'TEAM'] 1


In [70]:
driver = get_driver()

with driver.session() as session:
    result = session.run("""
        MATCH (n:__Entity__)
        RETURN
            labels(n) AS labels,
            n.name AS name
        ORDER BY labels(n), n.name
    """)

    for record in result:
        print(record["labels"], "->", record["name"])

driver.close()

['__Node__', '__Entity__', 'PERSON'] -> Alice Chen
['__Node__', '__Entity__', 'PERSON'] -> Bob Singh
['__Node__', '__Entity__', 'PERSON'] -> George Liu
['__Node__', '__Entity__', 'PERSON'] -> Hannah Brooks
['__Node__', '__Entity__', 'PROJECT'] -> project_atlas
['__Node__', '__Entity__', 'TEAM'] -> Search
['__Node__', '__Entity__', 'TECHNOLOGY'] -> Elasticsearch
['__Node__', '__Entity__', 'TECHNOLOGY'] -> Kafka
['__Node__', '__Entity__', 'TECHNOLOGY'] -> Kubernetes
['__Node__', '__Entity__', 'TECHNOLOGY'] -> Python


In [ ]:
#Inspect Relationships
driver = get_driver()

with driver.session() as session:

    result = session.run("""
        MATCH (a)-[r]->(b)
        WHERE NOT a:Chunk
        RETURN
            a.name AS source,
            type(r) AS relationship,
            b.name AS target
        ORDER BY source, relationship, target
    """)

    for record in result:
        print(
            f"{record['source']} "
            f"--{record['relationship']}--> "
            f"{record['target']}"
        )

driver.close()

Alice Chen --LEADS--> project_atlas
Alice Chen --WORKED_ON--> project_atlas
Bob Singh --WORKED_ON--> project_atlas
George Liu --WORKED_ON--> project_atlas
Hannah Brooks --WORKED_ON--> project_atlas
project_atlas --OWNED_BY--> Search
project_atlas --USES--> Elasticsearch
project_atlas --USES--> Kafka
project_atlas --USES--> Kubernetes
project_atlas --USES--> Python


In [ ]:
#Check neo4j status
import os
from urllib.parse import urlparse
from graph.neo4j_client import get_driver

DB = os.getenv("NEO4J_DATABASE", "neo4j")
URI = os.getenv("NEO4J_URI")

print("Database:", DB)
print("Neo4j host:", urlparse(URI).hostname)

driver = get_driver()

with driver.session(database=DB) as session:
    node_count = session.run(
        "MATCH (n) RETURN count(n) AS c"
    ).single()["c"]

    rel_count = session.run(
        "MATCH ()-[r]->() RETURN count(r) AS c"
    ).single()["c"]

    print("Nodes:", node_count)
    print("Relationships:", rel_count)

driver.close()

Database: neo4j
Neo4j host: 2725a5d2.databases.neo4j.io
Nodes: 114
Relationships: 206


In [74]:
driver = get_driver()

with driver.session(database=DB) as session:
    result = session.run("""
        MATCH (a:__Entity__)-[r]->(b:__Entity__)
        RETURN
            a.name AS source,
            type(r) AS relationship,
            b.name AS target
        ORDER BY source, relationship, target
    """)

    for row in result:
        print(
            f"{row['source']} "
            f"--{row['relationship']}--> "
            f"{row['target']}"
        )

driver.close()

Alice Chen --LEADS--> project_atlas
Alice Chen --WORKED_ON--> project_atlas
Bob Singh --WORKED_ON--> project_atlas
George Liu --WORKED_ON--> project_atlas
Hannah Brooks --WORKED_ON--> project_atlas
project_atlas --OWNED_BY--> Search
project_atlas --USES--> Elasticsearch
project_atlas --USES--> Kafka
project_atlas --USES--> Kubernetes
project_atlas --USES--> Python


INGEST 4 SAMPLE DOCS FROM CORPUS

In [ ]:
#Define the test batch
from pathlib import Path

test_files = [
    PROJECT_ROOT / "data/projects/project_atlas.md",
    PROJECT_ROOT / "data/people/person_alice.md",
    PROJECT_ROOT / "data/decisions/decision_d001.md",
    PROJECT_ROOT / "data/meetings/meeting_m001.md",
]

for f in test_files:
    print(f.name, "->", f.exists())

project_atlas.md -> True
person_alice.md -> True
decision_d001.md -> True
meeting_m001.md -> True


In [ ]:
#Load the test batch
from llama_index.core import SimpleDirectoryReader

batch_documents = SimpleDirectoryReader(
    input_files=[str(f) for f in test_files]
).load_data()

print("Documents loaded:", len(batch_documents))

for doc in batch_documents:
    print("-", doc.metadata.get("file_name"))

Documents loaded: 4
- project_atlas.md
- person_alice.md
- decision_d001.md
- meeting_m001.md


In [ ]:
#Clear graph for testing
from graph.neo4j_client import get_driver
import os

DB = os.getenv("NEO4J_DATABASE", "neo4j")

driver = get_driver()

with driver.session(database=DB) as session:
    session.run("""
        MATCH (n)
        DETACH DELETE n
    """)

driver.close()

print("✅ Test graph cleared")

✅ Test graph cleared


In [ ]:
#Run the four-document extraction
from llama_index.core import PropertyGraphIndex

index = PropertyGraphIndex.from_documents(
    batch_documents,
    kg_extractors=[kg_extractor],
    embed_model=embed_model,
    property_graph_store=graph_store,
    show_progress=True,
    use_async=False,
)

print("✅ Representative batch indexed")

Generating embeddings: 100%|██████████| 58/58 [00:00<00:00, 113.12it/s]


✅ Representative batch indexed


In [95]:
#Check Entity Counts
driver = get_driver()

with driver.session(database=DB) as session:
    result = session.run("""
        MATCH (n:__Entity__)
        RETURN labels(n) AS labels, count(*) AS count
        ORDER BY count DESC
    """)

    for row in result:
        print(row["labels"], row["count"])

driver.close()

['__Node__', '__Entity__', 'PERSON'] 5
['__Node__', '__Entity__', 'TECHNOLOGY', 'SKILL'] 3
['__Node__', '__Entity__', 'PROJECT'] 2
['__Node__', '__Entity__', 'TEAM'] 1
['__Node__', '__Entity__', 'TECHNOLOGY'] 1
['__Node__', '__Entity__', 'DECISION'] 1
['__Node__', '__Entity__', 'MEETING'] 1


In [96]:
#Inspect Relationships
driver = get_driver()

with driver.session(database=DB) as session:
    result = session.run("""
        MATCH (a:__Entity__)-[r]->(b:__Entity__)
        RETURN
            a.name AS source,
            type(r) AS relationship,
            b.name AS target
        ORDER BY relationship, source, target
    """)

    for row in result:
        print(
            f"{row['source']} "
            f"--{row['relationship']}--> "
            f"{row['target']}"
        )

driver.close()

Adopt Kafka for Atlas change feed --AFFECTS--> Project Atlas
Alice Chen --APPROVED--> Adopt Kafka for Atlas change feed
Alice Chen --ATTENDED--> Atlas architecture review
Carol Martinez --ATTENDED--> Atlas architecture review
George Liu --ATTENDED--> Atlas architecture review
Hannah Brooks --ATTENDED--> Atlas architecture review
Atlas architecture review --DISCUSSED--> Adopt Kafka for Atlas change feed
Alice Chen --HAS_SKILL--> Elasticsearch
Alice Chen --HAS_SKILL--> Kafka
Alice Chen --HAS_SKILL--> Python
Adopt Kafka for Atlas change feed --INVOLVES--> Kafka
Alice Chen --LEADS--> Project Atlas
Alice Chen --MEMBER_OF--> Search
Project Atlas --OWNED_BY--> Search
Carol Martinez --PROPOSED--> Adopt Kafka for Atlas change feed
Project Atlas --USES--> Elasticsearch
Project Atlas --USES--> Kafka
Project Atlas --USES--> Kubernetes
Project Atlas --USES--> Python
Alice Chen --WORKED_ON--> Project Atlas
Alice Chen --WORKED_ON--> Project Phoenix
Bob Singh --WORKED_ON--> Project Atlas
George Liu --

In [97]:
import os
from urllib.parse import urlparse
from graph.neo4j_client import get_driver

DB = os.getenv("NEO4J_DATABASE", "neo4j")
URI = os.getenv("NEO4J_URI")

print("Database:", DB)
print("Neo4j host:", urlparse(URI).hostname)

driver = get_driver()

with driver.session(database=DB) as session:
    node_count = session.run(
        "MATCH (n) RETURN count(n) AS c"
    ).single()["c"]

    rel_count = session.run(
        "MATCH ()-[r]->() RETURN count(r) AS c"
    ).single()["c"]

    print("Nodes:", node_count)
    print("Relationships:", rel_count)

driver.close()

Database: neo4j
Neo4j host: 2725a5d2.databases.neo4j.io
Nodes: 18
Relationships: 54


INGEST ENTIRE CORPUS

In [ ]:
#Building a corpus of all markdown files in the data folders
from pathlib import Path

DATA_FOLDERS = [
    PROJECT_ROOT / "data/people",
    PROJECT_ROOT / "data/projects",
    PROJECT_ROOT / "data/meetings",
    PROJECT_ROOT / "data/decisions",
    PROJECT_ROOT / "data/technical_docs",
]

all_files = []

for folder in DATA_FOLDERS:
    all_files.extend(sorted(folder.glob("*.md")))

print("Total corpus files:", len(all_files))

for folder in DATA_FOLDERS:
    count = len(list(folder.glob("*.md")))
    print(f"{folder.name:15s}: {count}")

Total corpus files: 33
people         : 10
projects       : 4
meetings       : 6
decisions      : 8
technical_docs : 5


In [99]:
#Load all documents from the corpus
from llama_index.core import SimpleDirectoryReader

all_documents = SimpleDirectoryReader(
    input_files=[str(f) for f in all_files]
).load_data()

print("Documents loaded:", len(all_documents))

Documents loaded: 33


In [ ]:
#Verify documents loaded
for doc in all_documents:
    print(doc.metadata.get("file_name"))

In [101]:
#Test Extractor Settings
print("Model:", llm.model)
print("Max tokens:", llm.max_tokens)
print("Max triplets:", kg_extractor.max_triplets_per_chunk)
print("Workers:", kg_extractor.num_workers)

Model: Qwen/Qwen3-30B-A3B-Instruct-2507
Max tokens: 2000
Max triplets: 15
Workers: 1


In [102]:
#Clear any test graph data from Neo4j
driver = get_driver()

with driver.session(database=DB) as session:
    session.run("""
        MATCH (n)
        DETACH DELETE n
    """)

driver.close()

print("✅ Test graph cleared")

✅ Test graph cleared


In [103]:
#Verify graph details after clearing
driver = get_driver()

with driver.session(database=DB) as session:
    count = session.run("""
        MATCH (n)
        RETURN count(n) AS count
    """).single()["count"]

print("Nodes:", count)

driver.close()

Nodes: 0


In [104]:
#Ingest First Batch of Documents
BATCH_SIZE = 5

document_batches = [
    all_documents[i:i + BATCH_SIZE]
    for i in range(0, len(all_documents), BATCH_SIZE)
]

print("Number of batches:", len(document_batches))

for i, batch in enumerate(document_batches, start=1):
    print(f"Batch {i}: {len(batch)} documents")

Number of batches: 7
Batch 1: 5 documents
Batch 2: 5 documents
Batch 3: 5 documents
Batch 4: 5 documents
Batch 5: 5 documents
Batch 6: 5 documents
Batch 7: 3 documents


In [105]:
#Process 1st batch of documents - Nebius Consumption
from llama_index.core import PropertyGraphIndex

index = PropertyGraphIndex.from_documents(
    document_batches[0],
    kg_extractors=[kg_extractor],
    embed_model=embed_model,
    property_graph_store=graph_store,
    show_progress=True,
    use_async=False,
)

print("✅ Batch 1 complete")

Generating embeddings: 100%|██████████| 56/56 [00:00<00:00, 136.52it/s]


✅ Batch 1 complete


In [106]:
#Verification
driver = get_driver()

with driver.session(database=DB) as session:

    nodes = session.run("""
        MATCH (n)
        RETURN count(n) AS count
    """).single()["count"]

    relationships = session.run("""
        MATCH ()-[r]->()
        RETURN count(r) AS count
    """).single()["count"]

print("Nodes:", nodes)
print("Relationships:", relationships)

driver.close()

Nodes: 26
Relationships: 61


In [107]:
#Process remaining batches of documents - Nebius Consumption
for batch_number, batch in enumerate(
    document_batches[1:],
    start=2
):
    print(
        f"\nProcessing batch {batch_number} "
        f"({len(batch)} documents)..."
    )

    PropertyGraphIndex.from_documents(
        batch,
        kg_extractors=[kg_extractor],
        embed_model=embed_model,
        property_graph_store=graph_store,
        show_progress=True,
        use_async=False,
    )

    print(f"✅ Batch {batch_number} complete")


Processing batch 2 (5 documents)...


Generating embeddings: 100%|██████████| 26/26 [00:00<00:00, 195.11it/s]


✅ Batch 2 complete

Processing batch 3 (5 documents)...


Generating embeddings: 100%|██████████| 10/10 [00:00<00:00, 116.30it/s]


✅ Batch 3 complete

Processing batch 4 (5 documents)...


...

✅ Batch 5 complete

Processing batch 6 (5 documents)...


Generating embeddings: 100%|██████████| 6/6 [00:00<00:00, 120.47it/s]


✅ Batch 6 complete

Processing batch 7 (3 documents)...


Generating embeddings: 100%|██████████| 23/23 [00:00<00:00, 117.85it/s]


✅ Batch 7 complete


In [108]:
#Inspect Final Node Distribution
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (n:__Entity__)
        RETURN
            [label IN labels(n)
             WHERE NOT label IN ['__Node__', '__Entity__']
            ] AS entity_type,
            count(*) AS count
        ORDER BY count DESC
    """)

    for row in result:
        print(
            row["entity_type"],
            row["count"]
        )

driver.close()

['DECISION'] 15
['SKILL'] 12
['PERSON'] 10
['MEETING'] 6
['TEAM'] 5
['PROJECT'] 4


In [109]:
#Duplicate name check
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (n:__Entity__)
        WITH labels(n) AS labels,
             n.name AS name,
             count(*) AS count
        WHERE count > 1
        RETURN labels, name, count
        ORDER BY count DESC
    """)

    rows = list(result)

    if not rows:
        print("✅ No duplicate entities")
    else:
        for row in rows:
            print(
                row["labels"],
                row["name"],
                row["count"]
            )

driver.close()

✅ No duplicate entities


In [110]:
#ID-like Names
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (n:__Entity__)
        WHERE n.name CONTAINS "_"
        RETURN
            labels(n) AS labels,
            n.name AS name
        ORDER BY name
    """)

    rows = list(result)

    if not rows:
        print("✅ All entity names are human-readable")
    else:
        print("⚠️ Possible ID-style names:")
        for row in rows:
            print(row["labels"], row["name"])

driver.close()

✅ All entity names are human-readable


In [111]:
#Save Raw DATA Extraction
import json
from pathlib import Path

driver = get_driver()

with driver.session(database=DB) as session:

    nodes = [
        {
            "labels": record["labels"],
            "properties": record["properties"],
        }
        for record in session.run("""
            MATCH (n)
            RETURN labels(n) AS labels,
                   properties(n) AS properties
        """)
    ]

    relationships = [
        {
            "source": record["source"],
            "source_labels": record["source_labels"],
            "relationship": record["relationship"],
            "target": record["target"],
            "target_labels": record["target_labels"],
        }
        for record in session.run("""
            MATCH (a)-[r]->(b)
            RETURN
                a.name AS source,
                labels(a) AS source_labels,
                type(r) AS relationship,
                b.name AS target,
                labels(b) AS target_labels
        """)
    ]

driver.close()

raw_graph = {
    "nodes": nodes,
    "relationships": relationships,
}

output_file = (
    PROJECT_ROOT /
    "evaluation" /
    "raw_extracted_graph.json"
)

output_file.write_text(
    json.dumps(raw_graph, indent=2, default=str)
)

print("Saved:", output_file)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/raw_extracted_graph.json


In [ ]:
#Load entity definitions
#We're going to use dataset_manifest.json for entity normalization only, not graph_ground_truth.json.
import json

manifest_path = (
    PROJECT_ROOT /
    "data" /
    "dataset_manifest.json"
)

manifest = json.loads(
    manifest_path.read_text()
)

CANONICAL_SKILLS = [
    x["name"]
    for x in manifest["skills"]
]

CANONICAL_TECHNOLOGIES = [
    x["name"]
    for x in manifest["technologies"]
]

CANONICAL_DECISIONS = [
    x["title"]
    for x in manifest["decisions"]
]

print("Skills:", CANONICAL_SKILLS)
print()
print("Technologies:", CANONICAL_TECHNOLOGIES)
print()
print("Decisions:", CANONICAL_DECISIONS)

Skills: ['Kafka', 'Kubernetes', 'Elasticsearch', 'Python', 'Redis', 'PostgreSQL', 'Airflow', 'Experimentation']

Technologies: ['Kafka', 'Kubernetes', 'Elasticsearch', 'Python', 'Redis', 'PostgreSQL', 'Airflow']

Decisions: ['Adopt Kafka for Atlas change feed', 'Standardize Elasticsearch mappings for Atlas', 'Deploy Atlas ranking services on Kubernetes', 'Use Redis for Phoenix online feature cache', 'Use Kafka as Phoenix behavior event bus', 'Use PostgreSQL for Nova investigation cases', 'Use Airflow for Mercury settlement workflows', 'Use PostgreSQL as Mercury reconciliation ledger']


In [ ]:
#Verify Expected counts
print(len(CANONICAL_SKILLS))
print(len(CANONICAL_TECHNOLOGIES))
print(len(CANONICAL_DECISIONS))

8
7
8


In [114]:
#Verify Real DEcicions
driver = get_driver()

with driver.session(database=DB) as session:

    existing = session.run("""
        MATCH (d:DECISION)
        WHERE d.name IN $canonical
        RETURN d.name AS name
    """,
    canonical=CANONICAL_DECISIONS)

    existing = {
        row["name"]
        for row in existing
    }

driver.close()

missing = (
    set(CANONICAL_DECISIONS)
    - existing
)

print("Missing canonical decisions:", missing)

Missing canonical decisions: set()


In [115]:
#Fix Skill Vs Technology Collision
driver = get_driver()

with driver.session(database=DB) as session:

    # Fix technologies used by projects
    session.run("""
        MATCH (p:PROJECT)-[r:USES]->(s:SKILL)

        MERGE (t:__Node__:__Entity__:TECHNOLOGY {
            id: "TECHNOLOGY::" +
                toLower(replace(s.name, " ", "-"))
        })

        SET t.name = s.name

        MERGE (p)-[:USES]->(t)

        DELETE r
    """)

    # Fix technologies involved in decisions
    session.run("""
        MATCH (d:DECISION)-[r:INVOLVES]->(s:SKILL)

        MERGE (t:__Node__:__Entity__:TECHNOLOGY {
            id: "TECHNOLOGY::" +
                toLower(replace(s.name, " ", "-"))
        })

        SET t.name = s.name

        MERGE (d)-[:INVOLVES]->(t)

        DELETE r
    """)

driver.close()

print("✅ Technology relationships normalized")

✅ Technology relationships normalized


In [116]:
#Remove Bogus Skills
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (s:SKILL)
        WHERE NOT s.name IN $canonical_skills

        WITH collect(s.name) AS removed,
             collect(s) AS nodes

        FOREACH (n IN nodes |
            DETACH DELETE n
        )

        RETURN removed
    """,
    canonical_skills=CANONICAL_SKILLS)

    row = result.single()

    print(
        "Removed:",
        row["removed"]
        if row else []
    )

driver.close()

Removed: ['Kafka architecture', 'Kubernetes production readiness', 'Elasticsearch relevance and mappings', 'payment settlement workflows']


In [117]:
#Remove Bogus Decisions
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (d:DECISION)
        WHERE NOT d.name IN $canonical_decisions

        WITH collect(d.name) AS removed,
             collect(d) AS nodes

        FOREACH (n IN nodes |
            DETACH DELETE n
        )

        RETURN removed
    """,
    canonical_decisions=CANONICAL_DECISIONS)

    row = result.single()

    print(
        "Removed:",
        row["removed"]
        if row else []
    )

driver.close()

Removed: ['Redis for the online feature cache', 'Kafka for the event bus', 'Kafka architecture decision', 'Elasticsearch architecture decision', 'Kubernetes architecture decision', 'Kafka decisions for Project Atlas', 'Kafka decisions for Project Phoenix']


In [118]:
#Recheck Graph Entity Counts
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (n:__Entity__)
        RETURN
            [
                label IN labels(n)
                WHERE NOT label IN [
                    '__Node__',
                    '__Entity__'
                ]
            ] AS entity_type,
            count(*) AS count

        ORDER BY entity_type
    """)

    for row in result:
        print(
            row["entity_type"],
            row["count"]
        )

driver.close()

['DECISION'] 8
['MEETING'] 6
['PERSON'] 10
['PROJECT'] 4
['SKILL'] 8
['TEAM'] 5
['TECHNOLOGY'] 7


In [119]:
#Confirm Entity Counts
expected_counts = {
    "PERSON": 10,
    "TEAM": 5,
    "PROJECT": 4,
    "SKILL": 8,
    "TECHNOLOGY": 7,
    "DECISION": 8,
    "MEETING": 6,
}

driver = get_driver()

with driver.session(database=DB) as session:
    rows = session.run("""
        MATCH (n:__Entity__)
        UNWIND [
            label IN labels(n)
            WHERE NOT label IN ['__Node__', '__Entity__']
        ] AS entity_type
        RETURN entity_type, count(*) AS count
    """)

    actual_counts = {
        row["entity_type"]: row["count"]
        for row in rows
    }

driver.close()

for entity_type, expected in expected_counts.items():
    actual = actual_counts.get(entity_type, 0)

    status = "✅" if actual == expected else "❌"

    print(
        f"{status} {entity_type:12s} "
        f"expected={expected}, actual={actual}"
    )

✅ PERSON       expected=10, actual=10
✅ TEAM         expected=5, actual=5
✅ PROJECT      expected=4, actual=4
✅ SKILL        expected=8, actual=8
✅ TECHNOLOGY   expected=7, actual=7
✅ DECISION     expected=8, actual=8
✅ MEETING      expected=6, actual=6


In [120]:
#Load document metada
import json

manifest_path = (
    PROJECT_ROOT /
    "data" /
    "dataset_manifest.json"
)

manifest = json.loads(
    manifest_path.read_text()
)

documents_metadata = manifest["documents"]

print("Documents:", len(documents_metadata))

Documents: 33


In [121]:
#Create Document Nodes
from pathlib import Path

driver = get_driver()

with driver.session(database=DB) as session:

    for doc in documents_metadata:

        document_id = doc["id"]
        path = doc["path"]
        doc_type = doc["type"]

        filename = Path(path).name

        session.run("""
            MERGE (d:__Node__:__Entity__:DOCUMENT {
                id: $id
            })

            SET
                d.name = $name,
                d.path = $path,
                d.doc_type = $doc_type
        """,
        id=document_id,
        name=filename,
        path=path,
        doc_type=doc_type)

driver.close()

print("✅ DOCUMENT nodes created")

✅ DOCUMENT nodes created


In [122]:
#Verify
driver = get_driver()

with driver.session(database=DB) as session:

    count = session.run("""
        MATCH (d:DOCUMENT)
        RETURN count(d) AS count
    """).single()["count"]

driver.close()

print("DOCUMENT nodes:", count)

DOCUMENT nodes: 33


In [123]:
#Add document -> Entity relationships
driver = get_driver()

with driver.session(database=DB) as session:

    for doc in documents_metadata:

        doc_id = doc["id"]
        doc_type = doc["type"]
        entity_id = doc.get("entity_id")

        if not entity_id:
            continue

        # Project overview
        if doc_type == "project_overview":

            project = next(
                (
                    p for p in manifest["projects"]
                    if p["id"] == entity_id
                ),
                None
            )

            if project:
                session.run("""
                    MATCH (d:DOCUMENT {id: $doc_id})
                    MATCH (p:PROJECT {name: $name})

                    MERGE (d)-[:DESCRIBES]->(p)
                """,
                doc_id=doc_id,
                name=project["name"])

        # Architecture decision
        elif doc_type == "architecture_decision":

            decision = next(
                (
                    d for d in manifest["decisions"]
                    if d["id"] == entity_id
                ),
                None
            )

            if decision:
                session.run("""
                    MATCH (doc:DOCUMENT {id: $doc_id})
                    MATCH (d:DECISION {name: $name})

                    MERGE (doc)-[:DOCUMENTS]->(d)
                """,
                doc_id=doc_id,
                name=decision["title"])

        # Person profile
        elif doc_type == "person_profile":

            person = next(
                (
                    p for p in manifest["people"]
                    if p["id"] == entity_id
                ),
                None
            )

            if person:
                session.run("""
                    MATCH (d:DOCUMENT {id: $doc_id})
                    MATCH (p:PERSON {name: $name})

                    MERGE (d)-[:MENTIONS]->(p)
                """,
                doc_id=doc_id,
                name=person["name"])

driver.close()

print("✅ Document relationships created")

✅ Document relationships created


In [124]:
#Final Entity Counts
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (n:__Entity__)

        UNWIND [
            label IN labels(n)
            WHERE NOT label IN ['__Node__', '__Entity__']
        ] AS entity_type

        RETURN
            entity_type,
            count(*) AS count

        ORDER BY entity_type
    """)

    total = 0

    for row in result:

        print(
            f"{row['entity_type']:12s}",
            row["count"]
        )

        total += row["count"]

driver.close()

print("\nTotal entities:", total)

DECISION     8
DOCUMENT     33
MEETING      6
PERSON       10
PROJECT      4
SKILL        8
TEAM         5
TECHNOLOGY   7

Total entities: 81


In [ ]:
#Define Business Relationships
BUSINESS_RELATIONSHIPS = [
    "MEMBER_OF",
    "HAS_SKILL",
    "WORKED_ON",
    "LEADS",
    "OWNED_BY",
    "USES",
    "PROPOSED",
    "APPROVED",
    "AFFECTS",
    "INVOLVES",
    "ATTENDED",
    "DISCUSSED",
    "DESCRIBES",
    "DOCUMENTS",
    "AUTHORED",
]
#Not counting LlamaIndex provenance as business relationships
#Excluded generic Chunk MENTIONS for now because LlamaIndex automatically generates those.
# Chunk --MENTIONS--> Entity

In [126]:
#Count business relationships
driver = get_driver()

with driver.session(database=DB) as session:

    result = session.run("""
        MATCH (a:__Entity__)-[r]->(b:__Entity__)

        WHERE type(r) IN $relationships

        RETURN
            type(r) AS relationship,
            count(*) AS count

        ORDER BY relationship
    """,
    relationships=BUSINESS_RELATIONSHIPS)

    total = 0

    for row in result:

        print(
            f"{row['relationship']:15s}",
            row["count"]
        )

        total += row["count"]

driver.close()

print("\nBusiness relationships:", total)

AFFECTS         8
APPROVED        8
ATTENDED        15
DESCRIBES       4
DISCUSSED       8
DOCUMENTS       8
HAS_SKILL       25
INVOLVES        8
LEADS           5
MEMBER_OF       10
OWNED_BY        4
PROPOSED        8
USES            15
WORKED_ON       17

Business relationships: 143


We have Fixed the Encountered Issues Now. The pipeline has gone through these stages - 33 source documents
        ↓
LlamaIndex + Qwen extraction
        ↓
raw graph in Neo4j
        ↓
we found extraction issues:
- SKILL/TECHNOLOGY collisions
- extra DECISION entities
- extra SKILL entities
        ↓
deterministic cleanup / canonicalization
        ↓
clean graph
        ↓
Cell 15 counts only meaningful business edges

EVALUATION AGAINST GROUND TRUTH

           Ground truth
                │
                │
                ▼
       ┌─────────────────┐
       │                 │
Expected entities    Expected edges
       │                 │
       └────────┬────────┘
                │
              compare
                │
       ┌────────▼────────┐
       │                 │
Neo4j entities       Neo4j edges
       │                 │
       └────────┬────────┘
                ▼
         Evaluation

In [127]:
#Load ground truth
import json

GROUND_TRUTH_PATH = (
    PROJECT_ROOT
    / "evaluation"
    / "graph_ground_truth.json"
)

ground_truth = json.loads(
    GROUND_TRUTH_PATH.read_text()
)

print("Expected nodes:", len(ground_truth["nodes"]))
print("Expected relationships:", len(ground_truth["edges"]))

Expected nodes: 81
Expected relationships: 174


In [128]:
#Build ID -> Entity Lookup
gt_nodes_by_id = {
    node["id"]: node
    for node in ground_truth["nodes"]
}

print(gt_nodes_by_id["person_alice"])
print(gt_nodes_by_id["project_atlas"])

{'id': 'person_alice', 'label': 'Person', 'name': 'Alice Chen', 'title': 'Principal Search Engineer'}
{'id': 'project_atlas', 'label': 'Project', 'name': 'Project Atlas', 'purpose': 'Modernize product search ranking with fresh behavioral signals and a unified retrieval stack.', 'status': 'Production'}


In [129]:
#Convert Expected Entites to (type,name)
def get_node_name(node):
    return (
        node.get("name")
        or node.get("title")
        or node.get("id")
    )


CORE_ENTITY_TYPES = {
    "Person",
    "Team",
    "Project",
    "Skill",
    "Technology",
    "Decision",
    "Meeting",
}


expected_entities = {
    (
        node["label"].upper(),
        get_node_name(node),
    )
    for node in ground_truth["nodes"]
    if node["label"] in CORE_ENTITY_TYPES
}

print("Expected core entities:", len(expected_entities))

Expected core entities: 48


In [130]:
#Pull actual entities from Neo4j
driver = get_driver()

with driver.session(database=DB) as session:

    rows = session.run("""
        MATCH (n:__Entity__)

        WHERE NOT n:DOCUMENT

        RETURN
            labels(n) AS labels,
            n.name AS name
    """)

    actual_entities = set()

    for row in rows:

        domain_labels = [
            label
            for label in row["labels"]
            if label not in {
                "__Node__",
                "__Entity__",
            }
        ]

        for label in domain_labels:
            actual_entities.add(
                (label.upper(), row["name"])
            )

driver.close()

print("Actual core entities:", len(actual_entities))

Actual core entities: 48


In [131]:
#Entity Precision/Recall
entity_tp = expected_entities & actual_entities

missing_entities = (
    expected_entities - actual_entities
)

unexpected_entities = (
    actual_entities - expected_entities
)

entity_precision = (
    len(entity_tp) / len(actual_entities)
    if actual_entities else 0
)

entity_recall = (
    len(entity_tp) / len(expected_entities)
    if expected_entities else 0
)

entity_f1 = (
    2 * entity_precision * entity_recall
    / (entity_precision + entity_recall)
    if entity_precision + entity_recall
    else 0
)

print(f"Entity precision: {entity_precision:.2%}")
print(f"Entity recall:    {entity_recall:.2%}")
print(f"Entity F1:        {entity_f1:.2%}")

print("\nMissing entities:")
for x in sorted(missing_entities):
    print(" -", x)

print("\nUnexpected entities:")
for x in sorted(unexpected_entities):
    print(" -", x)

Entity precision: 100.00%
Entity recall:    100.00%
Entity F1:        100.00%

Missing entities:

Unexpected entities:


In [ ]:
#Define core relationships
CORE_RELATIONSHIPS = {
    "MEMBER_OF",
    "HAS_SKILL",
    "WORKED_ON",
    "LEADS",
    "OWNED_BY",
    "USES",
    "PROPOSED",
    "APPROVED",
    "AFFECTS",
    "INVOLVES",
    "ATTENDED",
    "DISCUSSED",
}
#Excluding 
#DESCRIBES, DOCUMENTS, MENTIONS, AUTHORED because those involve document provenance

In [133]:
#Build Relationship Set
expected_relationships = set()

for edge in ground_truth["edges"]:

    if edge["type"] not in CORE_RELATIONSHIPS:
        continue

    source = gt_nodes_by_id[edge["source"]]
    target = gt_nodes_by_id[edge["target"]]

    expected_relationships.add(
        (
            source["label"].upper(),
            get_node_name(source),
            edge["type"],
            target["label"].upper(),
            get_node_name(target),
        )
    )

print(
    "Expected core relationships:",
    len(expected_relationships)
)

Expected core relationships: 136


In [134]:
#Pull actual relationships from Neo4j
driver = get_driver()

with driver.session(database=DB) as session:

    rows = session.run("""
        MATCH (a:__Entity__)-[r]->(b:__Entity__)

        WHERE type(r) IN $relationships

        RETURN
            labels(a) AS source_labels,
            a.name AS source_name,

            type(r) AS relationship,

            labels(b) AS target_labels,
            b.name AS target_name
    """,
    relationships=list(CORE_RELATIONSHIPS))

    actual_relationships = set()

    for row in rows:

        source_types = [
            label
            for label in row["source_labels"]
            if label not in {
                "__Node__",
                "__Entity__",
            }
        ]

        target_types = [
            label
            for label in row["target_labels"]
            if label not in {
                "__Node__",
                "__Entity__",
            }
        ]

        for source_type in source_types:
            for target_type in target_types:

                actual_relationships.add(
                    (
                        source_type.upper(),
                        row["source_name"],
                        row["relationship"],
                        target_type.upper(),
                        row["target_name"],
                    )
                )

driver.close()

print(
    "Actual core relationships:",
    len(actual_relationships)
)

Actual core relationships: 131


In [135]:
#Relationship Precision/Recall/F1
relationship_tp = (
    expected_relationships
    & actual_relationships
)

missing_relationships = (
    expected_relationships
    - actual_relationships
)

unexpected_relationships = (
    actual_relationships
    - expected_relationships
)

relationship_precision = (
    len(relationship_tp)
    / len(actual_relationships)
    if actual_relationships else 0
)

relationship_recall = (
    len(relationship_tp)
    / len(expected_relationships)
    if expected_relationships else 0
)

relationship_f1 = (
    2
    * relationship_precision
    * relationship_recall
    / (
        relationship_precision
        + relationship_recall
    )
    if relationship_precision + relationship_recall
    else 0
)

print(
    f"Relationship precision: "
    f"{relationship_precision:.2%}"
)

print(
    f"Relationship recall:    "
    f"{relationship_recall:.2%}"
)

print(
    f"Relationship F1:        "
    f"{relationship_f1:.2%}"
)

Relationship precision: 95.42%
Relationship recall:    91.91%
Relationship F1:        93.63%


In [136]:
#Missing relationships
print(
    f"Missing relationships: "
    f"{len(missing_relationships)}"
)

for rel in sorted(missing_relationships):
    print(
        f"{rel[1]} "
        f"--{rel[2]}--> "
        f"{rel[4]}"
    )

Missing relationships: 11
Alice Chen --ATTENDED--> Atlas relevance schema review
Alice Chen --LEADS--> Project Atlas
Bob Singh --ATTENDED--> Atlas relevance schema review
Carol Martinez --ATTENDED--> Mercury settlement design review
David Kim --LEADS--> Project Nova
Elena Rossi --ATTENDED--> Mercury settlement design review
Elena Rossi --LEADS--> Project Mercury
Farah Khan --LEADS--> Project Phoenix
George Liu --ATTENDED--> Atlas relevance schema review
Isaac Brown --ATTENDED--> Mercury settlement design review
Julia Patel --ATTENDED--> Mercury settlement design review


In [137]:
#Unexpected/Hallucinated relationships
print(
    f"Unexpected relationships: "
    f"{len(unexpected_relationships)}"
)

for rel in sorted(unexpected_relationships):
    print(
        f"{rel[1]} "
        f"--{rel[2]}--> "
        f"{rel[4]}"
    )

Unexpected relationships: 6
Carol Martinez --LEADS--> Project Atlas
Carol Martinez --LEADS--> Project Phoenix
Carol Martinez --WORKED_ON--> Project Atlas
Hannah Brooks --LEADS--> Project Atlas
Julia Patel --LEADS--> Project Mercury
Julia Patel --LEADS--> Project Phoenix


In [138]:
#Save Evaluation Results
evaluation_results = {
    "entity_metrics": {
        "precision": entity_precision,
        "recall": entity_recall,
        "f1": entity_f1,
        "expected": len(expected_entities),
        "actual": len(actual_entities),
        "correct": len(entity_tp),
    },

    "relationship_metrics": {
        "precision": relationship_precision,
        "recall": relationship_recall,
        "f1": relationship_f1,
        "expected": len(expected_relationships),
        "actual": len(actual_relationships),
        "correct": len(relationship_tp),
    },

    "missing_entities": [
        list(x)
        for x in sorted(missing_entities)
    ],

    "unexpected_entities": [
        list(x)
        for x in sorted(unexpected_entities)
    ],

    "missing_relationships": [
        list(x)
        for x in sorted(missing_relationships)
    ],

    "unexpected_relationships": [
        list(x)
        for x in sorted(unexpected_relationships)
    ],
}

In [139]:
evaluation_file = (
    PROJECT_ROOT
    / "evaluation"
    / "graph_extraction_results.json"
)

evaluation_file.write_text(
    json.dumps(
        evaluation_results,
        indent=2,
    )
)

print("Saved:", evaluation_file)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/graph_extraction_results.json


In [140]:
#Diagnostics to find missing and unexpected relationships
from collections import Counter

missing_by_type = Counter(
    rel[2]
    for rel in missing_relationships
)

unexpected_by_type = Counter(
    rel[2]
    for rel in unexpected_relationships
)

print("Missing by relationship type:")
for rel_type, count in missing_by_type.most_common():
    print(f"{rel_type:15s} {count}")

print("\nUnexpected by relationship type:")
for rel_type, count in unexpected_by_type.most_common():
    print(f"{rel_type:15s} {count}")

Missing by relationship type:
ATTENDED        7
LEADS           4

Unexpected by relationship type:
LEADS           5
WORKED_ON       1


In [141]:
print("=== MISSING ===")

for rel in sorted(missing_relationships):
    print(
        f"{rel[0]}:{rel[1]} "
        f"--{rel[2]}--> "
        f"{rel[3]}:{rel[4]}"
    )


print("\n=== UNEXPECTED ===")

for rel in sorted(unexpected_relationships):
    print(
        f"{rel[0]}:{rel[1]} "
        f"--{rel[2]}--> "
        f"{rel[3]}:{rel[4]}"
    )

=== MISSING ===
PERSON:Alice Chen --ATTENDED--> MEETING:Atlas relevance schema review
PERSON:Alice Chen --LEADS--> PROJECT:Project Atlas
PERSON:Bob Singh --ATTENDED--> MEETING:Atlas relevance schema review
PERSON:Carol Martinez --ATTENDED--> MEETING:Mercury settlement design review
PERSON:David Kim --LEADS--> PROJECT:Project Nova
PERSON:Elena Rossi --ATTENDED--> MEETING:Mercury settlement design review
PERSON:Elena Rossi --LEADS--> PROJECT:Project Mercury
PERSON:Farah Khan --LEADS--> PROJECT:Project Phoenix
PERSON:George Liu --ATTENDED--> MEETING:Atlas relevance schema review
PERSON:Isaac Brown --ATTENDED--> MEETING:Mercury settlement design review
PERSON:Julia Patel --ATTENDED--> MEETING:Mercury settlement design review

=== UNEXPECTED ===
PERSON:Carol Martinez --LEADS--> PROJECT:Project Atlas
PERSON:Carol Martinez --LEADS--> PROJECT:Project Phoenix
PERSON:Carol Martinez --WORKED_ON--> PROJECT:Project Atlas
PERSON:Hannah Brooks --LEADS--> PROJECT:Project Atlas
PERSON:Julia Patel --LEA

TEST BUILDING GRAPH RETRIEVAL


In [142]:
#Sanity-test our first GraphRAG Question
query = """
MATCH (p:PERSON)-[:WORKED_ON]->(project:PROJECT {name: $project}),
      (p)-[:HAS_SKILL]->(skill:SKILL {name: $skill})

RETURN
    p.name AS person,
    project.name AS project,
    skill.name AS skill
ORDER BY person
"""

driver = get_driver()

with driver.session(database=DB) as session:
    rows = list(
        session.run(
            query,
            project="Project Phoenix",
            skill="Kubernetes",
        )
    )

driver.close()

for row in rows:
    print(dict(row))

{'person': 'Bob Singh', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}
{'person': 'Hannah Brooks', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}


In [147]:
#Build reusable Neo4j query helper
def run_cypher(query, parameters=None):
    driver = get_driver()

    try:
        with driver.session(database=DB) as session:
            result = session.run(
                query,
                parameters or {}
            )

            return [
                dict(record)
                for record in result
            ]

    finally:
        driver.close()

In [143]:
#Define Graph Retrieval Intents
GRAPH_INTENTS = {
    "person_project_skill",
    "shared_projects",
    "team_projects",
    "leader_technologies",
    "decision_approvers",
    "expert_lookup",
    "multi_hop_decisions",
}

In [144]:
#Retrieval Templates
GRAPH_QUERIES = {

    "person_project_skill": """
        MATCH (p:PERSON)-[:WORKED_ON]->(project:PROJECT {name: $project}),
              (p)-[:HAS_SKILL]->(skill:SKILL {name: $skill})

        RETURN
            p.name AS person,
            project.name AS project,
            skill.name AS skill
        ORDER BY person
    """,

    "shared_projects": """
        MATCH (p:PERSON)-[:WORKED_ON]->(:PROJECT {name: $project1}),
              (p)-[:WORKED_ON]->(:PROJECT {name: $project2})

        RETURN DISTINCT
            p.name AS person

        ORDER BY person
    """,

    "team_projects": """
        MATCH (p:PERSON)-[:MEMBER_OF]->(team:TEAM {name: $team}),
              (p)-[:WORKED_ON]->(project:PROJECT)

        RETURN DISTINCT
            team.name AS team,
            p.name AS person,
            project.name AS project

        ORDER BY project, person
    """,

    "leader_technologies": """
        MATCH (leader:PERSON {name: $person})-[:LEADS]->(project:PROJECT),
              (project)-[:USES]->(technology:TECHNOLOGY)

        RETURN
            leader.name AS leader,
            project.name AS project,
            technology.name AS technology

        ORDER BY technology
    """,

    "decision_approvers": """
        MATCH (project:PROJECT {name: $project})-[:USES]->(technology:TECHNOLOGY),
              (decision:DECISION)-[:AFFECTS]->(project),
              (decision)-[:INVOLVES]->(technology),
              (person:PERSON)-[:APPROVED]->(decision)

        RETURN DISTINCT
            person.name AS approver,
            decision.name AS decision,
            technology.name AS technology

        ORDER BY decision
    """,
}

In [145]:
#Generaic Graph Retrieval Function
def retrieve_graph(intent, parameters):
    if intent not in GRAPH_QUERIES:
        raise ValueError(
            f"Unsupported graph intent: {intent}"
        )

    query = GRAPH_QUERIES[intent]

    rows = run_cypher(
        query,
        parameters
    )

    return {
        "intent": intent,
        "parameters": parameters,
        "results": rows,
    }

In [148]:
#Test 1
result = retrieve_graph(
    intent="person_project_skill",
    parameters={
        "project": "Project Phoenix",
        "skill": "Kubernetes",
    },
)

result

{'intent': 'person_project_skill',
 'parameters': {'project': 'Project Phoenix', 'skill': 'Kubernetes'},
 'results': [{'person': 'Bob Singh',
   'project': 'Project Phoenix',
   'skill': 'Kubernetes'},
  {'person': 'Hannah Brooks',
   'project': 'Project Phoenix',
   'skill': 'Kubernetes'}]}

BUILDING ACTUAL GraphRAG 

In [198]:
#graph_retriever.py
from pathlib import Path

graph_retriever_code = r'''
from typing import Any

from graph.neo4j_client import get_driver
from config import NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_URI

import os


DB = os.getenv("NEO4J_DATABASE", "neo4j")


GRAPH_QUERIES = {

 # Q1:
    # What database does Phoenix use for its online feature cache?
    "project_feature_technology": """
        MATCH (decision:DECISION)-[:AFFECTS]->(
            project:PROJECT {name: $project}
        )

        MATCH (decision)-[:INVOLVES]->(
            technology:TECHNOLOGY
        )

        WHERE toLower(decision.name)
              CONTAINS toLower($keyword)

        RETURN
            project.name AS project,
            decision.name AS decision,
            technology.name AS technology
    """,


    # Q2:
    # What is the purpose of Project Atlas?
    #
    # Use the graph to find the Project document,
    # then retrieve its source Chunk.
    "project_purpose": """
        MATCH (doc:DOCUMENT)-[:DESCRIBES]->(
            project:PROJECT {name: $project}
        )

        MATCH (chunk:Chunk)

        WHERE chunk.file_path ENDS WITH doc.path

        RETURN
            project.name AS project,
            doc.name AS document,
            chunk.file_name AS source_file,
            chunk.text AS evidence
    """,


    # Q3:
    # Why did Atlas adopt Kafka?
    #
    # Graph identifies the relevant decision.
    # Chunk provides the narrative rationale.
    "decision_rationale": """
        MATCH (decision:DECISION)-[:AFFECTS]->(
        project:PROJECT {name: $project}
    )

    MATCH (decision)-[:INVOLVES]->(
        technology:TECHNOLOGY {name: $technology}
    )

    MATCH (doc:DOCUMENT)-[:DOCUMENTS]->(decision)

    MATCH (chunk:Chunk)
    WHERE chunk.file_path ENDS WITH doc.path

    RETURN
        project.name AS project,
        decision.name AS decision,
        technology.name AS technology,
        doc.name AS document,
        chunk.file_name AS source_file,
        chunk.text AS evidence
    """,

    # Q4:
    # Who worked on Project Phoenix and also has Kubernetes experience?
    "person_project_skill": """
        MATCH (p:PERSON)-[:WORKED_ON]->(
            project:PROJECT {name: $project}
        ),
        (p)-[:HAS_SKILL]->(
            skill:SKILL {name: $skill}
        )

        RETURN
            p.name AS person,
            project.name AS project,
            skill.name AS skill

        ORDER BY person
    """,


    # Q5:
    # Which people worked on both Atlas and Phoenix?
    "shared_projects": """
        MATCH (p:PERSON)-[:WORKED_ON]->(
            project1:PROJECT {name: $project1}
        ),
        (p)-[:WORKED_ON]->(
            project2:PROJECT {name: $project2}
        )

        RETURN DISTINCT
            p.name AS person,
            project1.name AS project1,
            project2.name AS project2

        ORDER BY person
    """,


    # Q7:
    # Which projects were worked on by Search team members?
    "team_projects": """
        MATCH (p:PERSON)-[:MEMBER_OF]->(
            team:TEAM {name: $team}
        ),
        (p)-[:WORKED_ON]->(
            project:PROJECT
        )

        RETURN DISTINCT
            team.name AS team,
            p.name AS person,
            project.name AS project

        ORDER BY project, person
    """,


    # Q8:
    # Which technologies are used by projects led by Alice?
    "leader_technologies": """
        MATCH (
            leader:PERSON {name: $person}
        )-[:LEADS]->(
            project:PROJECT
        ),
        (project)-[:USES]->(
            technology:TECHNOLOGY
        )

        RETURN
            leader.name AS leader,
            project.name AS project,
            technology.name AS technology

        ORDER BY technology
    """,


    # Q6:
    # Who approved architecture decisions for technologies used by Atlas?
    "decision_approvers": """
        MATCH (
            project:PROJECT {name: $project}
        )-[:USES]->(
            technology:TECHNOLOGY
        )

        MATCH (
            decision:DECISION
        )-[:AFFECTS]->(
            project
        )

        MATCH (
            decision
        )-[:INVOLVES]->(
            technology
        )

        MATCH (
            person:PERSON
        )-[:APPROVED]->(
            decision
        )

        RETURN DISTINCT
            person.name AS approver,
            decision.name AS decision,
            technology.name AS technology,
            project.name AS project

        ORDER BY decision
    """,


    # Q9:
    # Who is the best Kafka expert based on project experience
    # and documented technical decisions?
    "expert_lookup": """
        MATCH (
            person:PERSON
        )-[:HAS_SKILL]->(
            skill:SKILL {name: $technology}
        )

        OPTIONAL MATCH (
            person
        )-[:WORKED_ON]->(
            project:PROJECT
        )-[:USES]->(
            tech:TECHNOLOGY {name: $technology}
        )

        OPTIONAL MATCH (
            person
        )-[:PROPOSED|APPROVED]->(
            decision:DECISION
        )-[:INVOLVES]->(
            decision_tech:TECHNOLOGY {name: $technology}
        )

        WITH
            person,
            collect(DISTINCT project.name) AS projects,
            collect(DISTINCT decision.name) AS decisions

        RETURN
            person.name AS person,
            projects,
            decisions,
            size(projects) AS project_count,
            size(decisions) AS decision_count,
            size(projects) + (2 * size(decisions)) AS expertise_score

        ORDER BY expertise_score DESC, person
    """,


    # Q10:
    # Decisions affecting projects owned by teams whose members
    # also worked on Phoenix.
    "multi_hop_decisions": """
        MATCH (
        member:PERSON
    )-[:WORKED_ON]->(
        reference_project:PROJECT {name: $project}
    )

    MATCH (
        member
    )-[:MEMBER_OF]->(
        team:TEAM
    )

    MATCH (
        affected_project:PROJECT
    )-[:OWNED_BY]->(
        team
    )

    MATCH (
        decision:DECISION
    )-[:AFFECTS]->(
        affected_project
    )

    RETURN
        decision.name AS decision,
        affected_project.name AS project,
        team.name AS team,
        collect(DISTINCT member.name) AS connecting_people

    ORDER BY decision
    """,
}


def run_cypher(
    query: str,
    parameters: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    """
    Execute a read-only Cypher query and return results
    as a list of dictionaries.
    """

    driver = get_driver()

    try:
        with driver.session(database=DB) as session:
            result = session.run(
                query,
                parameters or {},
            )

            return [
                dict(record)
                for record in result
            ]

    finally:
        driver.close()


def retrieve_graph(
    intent: str,
    parameters: dict[str, Any],
) -> dict[str, Any]:
    """
    Run one of the supported graph retrieval patterns.
    """

    if intent not in GRAPH_QUERIES:
        raise ValueError(
            f"Unsupported graph intent: {intent}. "
            f"Supported intents: {sorted(GRAPH_QUERIES)}"
        )

    rows = run_cypher(
        GRAPH_QUERIES[intent],
        parameters,
    )

    return {
        "intent": intent,
        "parameters": parameters,
        "result_count": len(rows),
        "results": rows,
    }
'''

output_file = (
    PROJECT_ROOT
    / "retrieval"
    / "graph_retriever.py"
)

output_file.write_text(graph_retriever_code)

print("Saved:", output_file)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/retrieval/graph_retriever.py


In [187]:
#Reloading the graph_retriever module
import importlib
import retrieval.graph_retriever

importlib.reload(
    retrieval.graph_retriever
)

from retrieval.graph_retriever import (
    retrieve_graph,
    run_cypher,
    GRAPH_QUERIES,
)

print("Supported intents:")

for intent in GRAPH_QUERIES:
    print("-", intent)

Supported intents:
- person_project_skill
- shared_projects
- team_projects
- leader_technologies
- decision_approvers
- expert_lookup
- multi_hop_decisions


In [151]:
#Test all retrieval patterns
graph_test_cases = [
    {
        "name": "Q4 - Project + Skill",
        "intent": "person_project_skill",
        "parameters": {
            "project": "Project Phoenix",
            "skill": "Kubernetes",
        },
    },
    {
        "name": "Q5 - Shared Projects",
        "intent": "shared_projects",
        "parameters": {
            "project1": "Project Atlas",
            "project2": "Project Phoenix",
        },
    },
    {
        "name": "Q6 - Decision Approvers",
        "intent": "decision_approvers",
        "parameters": {
            "project": "Project Atlas",
        },
    },
    {
        "name": "Q7 - Team Projects",
        "intent": "team_projects",
        "parameters": {
            "team": "Search",
        },
    },
    {
        "name": "Q8 - Leader Technologies",
        "intent": "leader_technologies",
        "parameters": {
            "person": "Alice Chen",
        },
    },
    {
        "name": "Q9 - Expert Lookup",
        "intent": "expert_lookup",
        "parameters": {
            "technology": "Kafka",
        },
    },
    {
        "name": "Q10 - Multi-hop Decisions",
        "intent": "multi_hop_decisions",
        "parameters": {
            "project": "Project Phoenix",
        },
    },
]


for test in graph_test_cases:

    print("=" * 80)
    print(test["name"])

    result = retrieve_graph(
        intent=test["intent"],
        parameters=test["parameters"],
    )

    print("Results:", result["result_count"])

    for row in result["results"]:
        print(row)

    print()

Q4 - Project + Skill
Results: 2
{'person': 'Bob Singh', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}
{'person': 'Hannah Brooks', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}

Q5 - Shared Projects
Results: 4
{'person': 'Alice Chen', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}
{'person': 'Bob Singh', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}
{'person': 'Carol Martinez', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}
{'person': 'Hannah Brooks', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}

Q6 - Decision Approvers
Results: 3
{'approver': 'Alice Chen', 'decision': 'Adopt Kafka for Atlas change feed', 'technology': 'Kafka', 'project': 'Project Atlas'}
{'approver': 'Alice Chen', 'decision': 'Deploy Atlas ranking services on Kubernetes', 'technology': 'Kubernetes', 'project': 'Project Atlas'}
{'approver': 'Alice Chen', 'decision': 'Standardize Elasticsearch mappings for Atlas', 'technology': 'Elasticsearch', 'project

In [152]:
#Create query planner
from pathlib import Path

planner_code = r'''
import json
from pathlib import Path

from config import ROOT_DIR


# ---------------------------------------------------------
# Load known organizational entities
# ---------------------------------------------------------

MANIFEST_PATH = ROOT_DIR / "data" / "dataset_manifest.json"

_manifest = json.loads(
    MANIFEST_PATH.read_text()
)

KNOWN_PROJECTS = [
    item["name"]
    for item in _manifest["projects"]
]

KNOWN_PEOPLE = [
    item["name"]
    for item in _manifest["people"]
]

KNOWN_TEAMS = [
    item["name"]
    for item in _manifest["teams"]
]

KNOWN_SKILLS = [
    item["name"]
    for item in _manifest["skills"]
]

KNOWN_TECHNOLOGIES = [
    item["name"]
    for item in _manifest["technologies"]
]

# A query might refer to Kafka as either a skill or technology.
KNOWN_CAPABILITIES = sorted(
    set(KNOWN_SKILLS + KNOWN_TECHNOLOGIES)
)


def _find_all(text: str, candidates: list[str]) -> list[str]:
    """
    Return all candidate entity names explicitly mentioned
    in the question.
    """

    text_lower = text.lower()

    return [
        candidate
        for candidate in candidates
        if candidate.lower() in text_lower
    ]


def plan_graph_query(question: str) -> dict:
    """
    Convert supported natural-language questions into
    a graph intent and parameter dictionary.

    No LLM is used here.
    """

    q = question.lower()

    projects = _find_all(
        question,
        KNOWN_PROJECTS,
    )

    people = _find_all(
        question,
        KNOWN_PEOPLE,
    )

    teams = _find_all(
        question,
        KNOWN_TEAMS,
    )

    capabilities = _find_all(
        question,
        KNOWN_CAPABILITIES,
    )

    # -----------------------------------------------------
    # Q10 — Multi-hop decisions
    # Put before simpler "decision" rules.
    # -----------------------------------------------------

    if (
        "decision" in q
        and "owned" in q
        and "team" in q
        and len(projects) >= 1
    ):
        return {
            "intent": "multi_hop_decisions",
            "parameters": {
                "project": projects[0],
            },
        }

    # -----------------------------------------------------
    # Q4 — person + project + skill
    # -----------------------------------------------------

    if (
        len(projects) >= 1
        and len(capabilities) >= 1
        and "worked" in q
        and (
            "experience" in q
            or "skill" in q
            or "knows" in q
        )
    ):
        return {
            "intent": "person_project_skill",
            "parameters": {
                "project": projects[0],
                "skill": capabilities[0],
            },
        }

    # -----------------------------------------------------
    # Q5 — people who worked on two projects
    # -----------------------------------------------------

    if (
        len(projects) >= 2
        and "worked" in q
    ):
        return {
            "intent": "shared_projects",
            "parameters": {
                "project1": projects[0],
                "project2": projects[1],
            },
        }

    # -----------------------------------------------------
    # Q6 — decision approvers
    # -----------------------------------------------------

    if (
        "approv" in q
        and "decision" in q
        and len(projects) >= 1
    ):
        return {
            "intent": "decision_approvers",
            "parameters": {
                "project": projects[0],
            },
        }

    # -----------------------------------------------------
    # Q7 — projects connected to a team
    # -----------------------------------------------------

    if (
        len(teams) >= 1
        and "project" in q
        and (
            "member" in q
            or "team" in q
        )
    ):
        return {
            "intent": "team_projects",
            "parameters": {
                "team": teams[0],
            },
        }

    # -----------------------------------------------------
    # Q8 — technologies used by projects led by a person
    # -----------------------------------------------------

    if (
        len(people) >= 1
        and "technolog" in q
        and (
            "led by" in q
            or "leads" in q
        )
    ):
        return {
            "intent": "leader_technologies",
            "parameters": {
                "person": people[0],
            },
        }

    # -----------------------------------------------------
    # Q9 — expertise lookup
    # -----------------------------------------------------

    if (
        len(capabilities) >= 1
        and (
            "best person" in q
            or "consult" in q
            or "expert" in q
        )
    ):
        return {
            "intent": "expert_lookup",
            "parameters": {
                "technology": capabilities[0],
            },
        }

    # -----------------------------------------------------
    # Unsupported graph question
    # -----------------------------------------------------

    return {
        "intent": None,
        "parameters": {},
    }
'''

planner_path = (
    PROJECT_ROOT
    / "retrieval"
    / "query_planner.py"
)

planner_path.write_text(planner_code)

print("Saved:", planner_path)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/retrieval/query_planner.py


In [153]:
#Import and test the planner
import importlib
import retrieval.query_planner

importlib.reload(
    retrieval.query_planner
)

from retrieval.query_planner import (
    plan_graph_query,
)

In [154]:
#Test 1
question = (
    "Who worked on Project Phoenix "
    "and also has Kubernetes experience?"
)

plan_graph_query(question)

{'intent': 'person_project_skill',
 'parameters': {'project': 'Project Phoenix', 'skill': 'Kubernetes'}}

In [155]:
#Test 2
plan_graph_query(
    "Which technologies are used by "
    "projects led by Alice Chen?"
)

{'intent': 'leader_technologies', 'parameters': {'person': 'Alice Chen'}}

In [156]:
#Connecting the query planner and graph retriever
from retrieval.graph_retriever import retrieve_graph


def graph_search(question: str) -> dict:

    plan = plan_graph_query(question)

    if plan["intent"] is None:

        return {
            "question": question,
            "route": None,
            "parameters": {},
            "result_count": 0,
            "results": [],
            "error": "No supported graph retrieval intent found.",
        }

    retrieval = retrieve_graph(
        intent=plan["intent"],
        parameters=plan["parameters"],
    )

    return {
        "question": question,
        "route": plan["intent"],
        "parameters": plan["parameters"],
        "result_count": retrieval["result_count"],
        "results": retrieval["results"],
        "error": None,
    }

In [ ]:
#Test graph search
graph_search(
    "Who worked on Project Phoenix "
    "and also has Kubernetes experience?"
)

{'question': 'Who worked on Project Phoenix and also has Kubernetes experience?',
 'route': 'person_project_skill',
 'parameters': {'project': 'Project Phoenix', 'skill': 'Kubernetes'},
 'result_count': 2,
 'results': [{'person': 'Bob Singh',
   'project': 'Project Phoenix',
   'skill': 'Kubernetes'},
  {'person': 'Hannah Brooks',
   'project': 'Project Phoenix',
   'skill': 'Kubernetes'}],
 'error': None}

In [158]:
#Test All Q4-Q10 Questions
import json

questions_path = (
    PROJECT_ROOT
    / "evaluation"
    / "questions.json"
)

evaluation_questions = json.loads(
    questions_path.read_text()
)

graph_questions = [
    q
    for q in evaluation_questions
    if q["id"] >= 4
]

print(
    "Graph-heavy questions:",
    len(graph_questions)
)

Graph-heavy questions: 7


In [193]:
graph_retrieval_results = []

for item in graph_questions:

    result = graph_search(
        item["question"]
    )

    graph_retrieval_results.append({
        "id": item["id"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        **result,
    })

    print("=" * 90)
    print(f"Q{item['id']}")
    print(item["question"])

    print("\nRoute:")
    print(result["route"])

    print("\nParameters:")
    print(result["parameters"])

    print("\nRetrieved evidence:")

    for row in result["results"]:
        print(" ", row)

    print()

Q4
Who worked on Project Phoenix and also has Kubernetes experience?

Route:
person_project_skill

Parameters:
{'project': 'Project Phoenix', 'skill': 'Kubernetes'}

Retrieved evidence:
  {'person': 'Bob Singh', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}
  {'person': 'Hannah Brooks', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}

Q5
Which people worked on both Project Atlas and Project Phoenix?

Route:
shared_projects

Parameters:
{'project1': 'Project Atlas', 'project2': 'Project Phoenix'}

Retrieved evidence:
  {'person': 'Alice Chen', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}
  {'person': 'Bob Singh', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}
  {'person': 'Carol Martinez', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}
  {'person': 'Hannah Brooks', 'project1': 'Project Atlas', 'project2': 'Project Phoenix'}

Q6
Who approved the architecture decisions for technologies used by Project Atlas?

Route:
decision_approvers



In [160]:
#Save Graph Rag retrieval results
output_path = (
    PROJECT_ROOT
    / "evaluation"
    / "graph_retrieval_results.json"
)

output_path.write_text(
    json.dumps(
        graph_retrieval_results,
        indent=2,
        default=str,
    )
)

print("Saved:", output_path)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/graph_retrieval_results.json


Working of current GraphRAG:


1.knowledge graph ingestion
2.canonicalization
3.Neo4j graph
4.graph extraction evaluation
5.deterministic plan_graph_query()
6.parameterized Cypher templates
7.graph retrieval tests

In [ ]:
from retrieval.graph_retriever import GRAPH_QUERIES

print("Graph intents:", len(GRAPH_QUERIES))

for intent in GRAPH_QUERIES:
    print("-", intent)

Graph intents: 10
- person_project_skill
- shared_projects
- team_projects
- leader_technologies
- decision_approvers
- expert_lookup
- multi_hop_decisions
- project_feature_technology
- project_purpose
- decision_rationale


In [169]:
#Test Q1 - Q3
q1_result = retrieve_graph(
    intent="project_feature_technology",
    parameters={
        "project": "Project Phoenix",
        "keyword": "online feature cache",
    },
)

for row in q1_result["results"]:
    print(row)


{'project': 'Project Phoenix', 'decision': 'Use Redis for Phoenix online feature cache', 'technology': 'Redis'}


In [170]:
q2_result = retrieve_graph(
    intent="project_purpose",
    parameters={
        "project": "Project Atlas",
    },
)

for row in q2_result["results"]:
    print(row)


{'project': 'Project Atlas', 'document': 'project_atlas.md', 'source_file': 'project_atlas.md', 'evidence': '---\ndoc_id: doc_project_atlas\ndoc_type: project_overview\nentity_id: project_atlas\nowner: Search\n---\n# Project Atlas\n\n**Status:** Production  \n**Owning team:** Search  \n**Project lead:** Alice Chen  \n**Core technologies:** Kafka, Elasticsearch, Python, Kubernetes\n\n## Purpose\nModernize product search ranking with fresh behavioral signals and a unified retrieval stack.\n\n## Contributors\nAlice Chen, Bob Singh, George Liu, Hannah Brooks'}


In [171]:
q3_result = retrieve_graph(
    intent="decision_rationale",
    parameters={
        "project": "Project Atlas",
        "technology": "Kafka",
    },
)

for row in q3_result["results"]:
    print(row)

{'project': 'Project Atlas', 'decision': 'Adopt Kafka for Atlas change feed', 'technology': 'Kafka', 'document': 'decision_d001.md', 'source_file': 'decision_d001.md', 'evidence': '---\ndoc_id: doc_decision_d001\ndoc_type: architecture_decision\nentity_id: decision_d001\nproject_id: project_atlas\ndate: 2026-02-12\n---\n# Adopt Kafka for Atlas change feed\n\n**Decision ID:** decision_d001  \n**Project:** Project Atlas  \n**Technology:** Kafka  \n**Proposed by:** Carol Martinez  \n**Approved by:** Alice Chen  \n**Date:** 2026-02-12\n\n## Decision\nAdopt Kafka for Atlas change feed.\n\n## Rationale\nAtlas needed replayable, ordered product and behavior updates. Kafka provided durable event history and decoupled producers from ranking consumers.\n\n## Alternatives considered\nDirect database polling, Managed queue without replay'}


In [172]:
#Extend the deterministic planner for Q1–Q3
from retrieval.query_planner import plan_graph_query

def plan_graph_query_v2(question: str):

    q = question.lower()

    # Q1
    if (
        "project phoenix" in q
        and "feature cache" in q
    ):
        return {
            "intent": "project_feature_technology",
            "parameters": {
                "project": "Project Phoenix",
                "keyword": "online feature cache",
            },
        }

    # Q2
    if (
        "purpose" in q
        and "project atlas" in q
    ):
        return {
            "intent": "project_purpose",
            "parameters": {
                "project": "Project Atlas",
            },
        }

    # Q3
    if (
        "project atlas" in q
        and "kafka" in q
        and (
            "why" in q
            or "adopt" in q
        )
    ):
        return {
            "intent": "decision_rationale",
            "parameters": {
                "project": "Project Atlas",
                "technology": "Kafka",
            },
        }

    # Q4–Q10
    return plan_graph_query(question)

In [173]:
#Update graph search
def graph_search(question: str):

    plan = plan_graph_query_v2(question)

    if plan["intent"] is None:

        return {
            "question": question,
            "route": None,
            "parameters": {},
            "result_count": 0,
            "results": [],
            "error": "No supported graph intent found.",
        }

    retrieval = retrieve_graph(
        intent=plan["intent"],
        parameters=plan["parameters"],
    )

    return {
        "question": question,
        "route": plan["intent"],
        "parameters": plan["parameters"],
        "result_count": retrieval["result_count"],
        "results": retrieval["results"],
        "error": None,
    }

In [195]:
#Run all 10 questions through the updated graph search
graph_retrieval_results = []

for item in evaluation_questions:

    result = graph_search(
        item["question"]
    )

    graph_retrieval_results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "route": result["route"],
        "parameters": result["parameters"],
        "retrieval": result["results"],
    })

    print("=" * 100)

    print(
        f"Q{item['id']} "
        f"[{item['category']}]"
    )

    print(item["question"])

    print("\nGraph route:")
    print(result["route"])

    print("\nEvidence:")

    for row in result["results"]:
        print(row)

    print()

Q1 [single_fact]
What database does Project Phoenix use for its online feature cache?

Graph route:
project_feature_technology

Evidence:
{'project': 'Project Phoenix', 'decision': 'Use Redis for Phoenix online feature cache', 'technology': 'Redis'}

Q2 [single_fact]
What is the purpose of Project Atlas?

Graph route:
project_purpose

Evidence:
{'project': 'Project Atlas', 'document': 'project_atlas.md', 'source_file': 'project_atlas.md', 'evidence': '---\ndoc_id: doc_project_atlas\ndoc_type: project_overview\nentity_id: project_atlas\nowner: Search\n---\n# Project Atlas\n\n**Status:** Production  \n**Owning team:** Search  \n**Project lead:** Alice Chen  \n**Core technologies:** Kafka, Elasticsearch, Python, Kubernetes\n\n## Purpose\nModernize product search ranking with fresh behavioral signals and a unified retrieval stack.\n\n## Contributors\nAlice Chen, Bob Singh, George Liu, Hannah Brooks'}

Q3 [semantic]
Why did Project Atlas adopt Kafka?

Graph route:
decision_rationale

Eviden

In [175]:
#Check Graph Coverage
for item in graph_retrieval_results:

    status = (
        "✅"
        if item["route"]
        and len(item["retrieval"]) > 0
        else "❌"
    )

    print(
        status,
        f"Q{item['id']}",
        item["route"],
        f"results={len(item['retrieval'])}"
    )

✅ Q1 project_feature_technology results=1
✅ Q2 project_purpose results=1
✅ Q3 decision_rationale results=1
✅ Q4 person_project_skill results=2
✅ Q5 shared_projects results=4
✅ Q6 decision_approvers results=3
✅ Q7 team_projects results=3
❌ Q8 leader_technologies results=0
✅ Q9 expert_lookup results=5
✅ Q10 multi_hop_decisions results=9


In [ ]:
#Check why Q8 failed
run_cypher("""
    MATCH (p:PERSON {name: "Alice Chen"})-[r]->(project:PROJECT)
    RETURN
        p.name AS person,
        type(r) AS relationship,
        project.name AS project
""")

run_cypher("""
    MATCH (:PERSON {name: "Alice Chen"})-[r:LEADS]->(p:PROJECT)
    RETURN p.name AS project
""")# It returns empty because the LEADS edge is absent. 

[]

In [178]:
run_cypher("""
    MATCH (p:PROJECT {name: "Project Atlas"})-[:USES]->(t:TECHNOLOGY)
    RETURN t.name AS technology
    ORDER BY technology
""")

[{'technology': 'Elasticsearch'},
 {'technology': 'Kafka'},
 {'technology': 'Kubernetes'},
 {'technology': 'Python'}]

In [196]:
#Save the final graph retrieval results benchmark
graph_output_path = (
    PROJECT_ROOT
    / "evaluation"
    / "graph_retrieval_results.json"
)

graph_output_path.write_text(
    json.dumps(
        graph_retrieval_results,
        indent=2,
        default=str,
    )
)

print("Saved:", graph_output_path)

Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/graph_retrieval_results.json


In [180]:
#Validate why it stopped
run_cypher("""
    MATCH (:PERSON {name: "Alice Chen"})-[r:MEMBER_OF]->(t:TEAM)
    RETURN t.name AS team
""")

[{'team': 'Search'}]

In [197]:
run_cypher("""
    MATCH (:PERSON {name: "Alice Chen"})-[r:WORKED_ON]->(p:PROJECT)
    RETURN p.name AS project
    ORDER BY project
""")

[{'project': 'Project Atlas'}, {'project': 'Project Phoenix'}]

In [188]:
q10_check = retrieve_graph(
    intent="multi_hop_decisions",
    parameters={
        "project": "Project Phoenix"
    },
)

for row in q10_check["results"]:
    print(row)

{'decision': 'Adopt Kafka for Atlas change feed', 'project': 'Project Atlas', 'team': 'Search', 'connecting_people': ['Alice Chen']}
{'decision': 'Deploy Atlas ranking services on Kubernetes', 'project': 'Project Atlas', 'team': 'Search', 'connecting_people': ['Alice Chen']}
{'decision': 'Standardize Elasticsearch mappings for Atlas', 'project': 'Project Atlas', 'team': 'Search', 'connecting_people': ['Alice Chen']}
{'decision': 'Use Airflow for Mercury settlement workflows', 'project': 'Project Mercury', 'team': 'Payments', 'connecting_people': ['Julia Patel']}
{'decision': 'Use Kafka as Phoenix behavior event bus', 'project': 'Project Phoenix', 'team': 'Recommendations', 'connecting_people': ['Farah Khan', 'Bob Singh']}
{'decision': 'Use PostgreSQL as Mercury reconciliation ledger', 'project': 'Project Mercury', 'team': 'Payments', 'connecting_people': ['Julia Patel']}
{'decision': 'Use Redis for Phoenix online feature cache', 'project': 'Project Phoenix', 'team': 'Recommendations', 

In [183]:
q10_saved = graph_retrieval_results[9]

print("Saved Q10 evidence:")
for row in q10_saved["retrieval"]:
    print(row)

Saved Q10 evidence:
{'decision': 'Adopt Kafka for Atlas change feed', 'project': 'Project Atlas', 'team': 'Search', 'connecting_person': 'Alice Chen'}
{'decision': 'Deploy Atlas ranking services on Kubernetes', 'project': 'Project Atlas', 'team': 'Search', 'connecting_person': 'Alice Chen'}
{'decision': 'Standardize Elasticsearch mappings for Atlas', 'project': 'Project Atlas', 'team': 'Search', 'connecting_person': 'Alice Chen'}
{'decision': 'Use Airflow for Mercury settlement workflows', 'project': 'Project Mercury', 'team': 'Payments', 'connecting_person': 'Julia Patel'}
{'decision': 'Use Kafka as Phoenix behavior event bus', 'project': 'Project Phoenix', 'team': 'Recommendations', 'connecting_person': 'Farah Khan'}
{'decision': 'Use Kafka as Phoenix behavior event bus', 'project': 'Project Phoenix', 'team': 'Recommendations', 'connecting_person': 'Bob Singh'}
{'decision': 'Use PostgreSQL as Mercury reconciliation ledger', 'project': 'Project Mercury', 'team': 'Payments', 'connectin

In [192]:
#Confirm Unique decisions
q10_saved = graph_retrieval_results[9]

print("Rows:", len(q10_saved["retrieval"]))

unique_decisions = sorted({
    row["decision"]
    for row in q10_saved["retrieval"]
})

print("Unique decisions:", len(unique_decisions))

for decision in unique_decisions:
    print("-", decision)

Rows: 9
Unique decisions: 7
- Adopt Kafka for Atlas change feed
- Deploy Atlas ranking services on Kubernetes
- Standardize Elasticsearch mappings for Atlas
- Use Airflow for Mercury settlement workflows
- Use Kafka as Phoenix behavior event bus
- Use PostgreSQL as Mercury reconciliation ledger
- Use Redis for Phoenix online feature cache
